# 02. Deduplicar Splink + avaliar vs coorte


In [ ]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

import pandas as pd
from config import OUTPUT_DIR, get_connection, print_paths, require_tables

print_paths()
con = get_connection()
require_tables(con, ['registro_unificado', 'ground_truth_clusters'], notebook_origem='00')

df = con.execute('''
    SELECT r.*, g.cluster FROM registro_unificado r
    LEFT JOIN ground_truth_clusters g ON r.unique_id = g.unique_id
''').df()
print('Registros:', len(df))


In [ ]:
from splink import DuckDBAPI, Linker, SettingsCreator, block_on
import splink.comparison_library as cl

db_api = DuckDBAPI()
blocking_rules = [
    block_on('substr(primeiro_nome,1,3)', 'substr(ultimo_nome,1,4)'),
    block_on('ultimo_nome', 'data_nascimento'),
    block_on('primeiro_nome', 'data_nascimento'),
    block_on('substr(cep,1,5)', 'primeiro_nome'),
    block_on('nome_mae', 'data_nascimento'),
]
comparisons = [
    cl.NameComparison('nome_completo'),
    cl.NameComparison('primeiro_nome').configure(term_frequency_adjustments=True),
    cl.NameComparison('ultimo_nome').configure(term_frequency_adjustments=True),
    cl.DateOfBirthComparison('data_nascimento', input_is_string=True),
    cl.NameComparison('nome_mae'),
    cl.ExactMatch('sexo').configure(term_frequency_adjustments=True),
    cl.ExactMatch('uf').configure(term_frequency_adjustments=True),
]
settings = SettingsCreator(
    link_type='dedupe_only',
    unique_id_column_name='unique_id',
    comparisons=comparisons,
    blocking_rules_to_generate_predictions=blocking_rules,
)
linker = Linker(df, settings, db_api=db_api)


In [ ]:
deterministic_rules = [
    block_on('primeiro_nome', 'ultimo_nome', 'data_nascimento'),
    block_on('nome_mae', 'data_nascimento'),
]
linker.training.estimate_probability_two_random_records_match(deterministic_rules, recall=0.8)
linker.training.estimate_u_using_random_sampling(max_pairs=1_000_000)
linker.training.estimate_parameters_using_expectation_maximisation(
    block_on('primeiro_nome', 'ultimo_nome', 'data_nascimento')
)


In [ ]:
predictions = linker.inference.predict(threshold_match_probability=0.5)
df_predictions = predictions.as_pandas_dataframe()
clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    predictions, threshold_match_probability=0.95,
)
df_clusters = clusters.as_pandas_dataframe()
print('Pares:', len(df_predictions), 'Clusters:', df_clusters['cluster_id'].nunique())


In [ ]:
pairs_gt = con.execute('''
    SELECT g1.unique_id AS id_censo, g2.unique_id AS id_cpf
    FROM ground_truth_clusters g1
    JOIN ground_truth_clusters g2 ON g1.cluster = g2.cluster
    WHERE g1.unique_id LIKE 'censo_%' AND g2.unique_id LIKE 'cpf_%' AND g1.cluster LIKE 'gt_%'
''').df()

pred_map = df_clusters.set_index('unique_id')['cluster_id'].to_dict()
hits = sum(
    1 for _, row in pairs_gt.iterrows()
    if pred_map.get(row['id_censo']) == pred_map.get(row['id_cpf'])
    and pred_map.get(row['id_censo']) is not None
)
recall = hits / len(pairs_gt) if len(pairs_gt) else 0.0
print(f'Recall cross-source: {hits}/{len(pairs_gt)} = {recall:.3f}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_predictions.to_parquet(OUTPUT_DIR / 'splink_predictions.parquet')
df_clusters.to_parquet(OUTPUT_DIR / 'splink_clusters.parquet')
pd.DataFrame([{'recall_cross_source': recall, 'n_pares_gt': len(pairs_gt), 'hits': hits}]).to_csv(
    OUTPUT_DIR / 'metricas_cohort.csv', index=False
)
con.close()
